The goal here is to do a *single* forward model that could be compared to a single MSA exposure.  The intent here is to work out all the steps, get rough timing, and use that to determine which need further optimization.  All relevant notebooks start with `single_forward_model`.

This comes after the PSF grid was generated in [single_forward_model3_psfgrid.ipynb](single_forward_model3_psfgrid.ipynb)

In [1]:
import time
import pickle
from pathlib import Path

import numpy as np

from matplotlib import pyplot as plt

from astropy import visualization as viz
viz.quantity_support()
from jwst import datamodels

from astropy import units as u
from astropy.coordinates import SkyCoord, SpectralCoord

from photutils import psf

from tqdm.auto import tqdm


# Load Data and Information

In [2]:
n6791fits = list(Path('../ngc6791_cals').glob('*.fits'))

dm0 = datamodels.open(n6791fits[0])

slit = dm0.slits[49]
scoord = SkyCoord(slit.source_ra, slit.source_dec, unit='deg')
swcs = slit.meta.wcs

assert slit.source_name == '2609_21'

In [3]:
apogee_id = '2M19210086+3745339'
apogee_vhelio_avg = -45.42058
apogee_vunc = 0.1100559
apogee_teff = 4416.742
apogee_logg = 2.269546
apogee_m_h = 0.34755

apogee_feh = np.log10(apogee_m_h)

In [4]:
with open('single_forward_model_model_spec.pkl', 'rb') as f:
    data = pickle.load(f)

model_wave = data['wave']
model_freq = model_wave.to(u.Hz, equivalencies=u.spectral())
model_flux = data['model']
model_path = data['phoenix_path']

# note that the normalization is at the "surface" of the star, so its crazy high - renormalize as needed
model_flux_fnu = model_flux.to(u.MJy, equivalencies=u.spectral_density(model_wave)) 

del data

In [5]:
with open('single_forward_model_psf_grid_os9_fov20.pkl', 'rb') as f:
    data = pickle.load(f)

psf_interpolator = data['psf_grid']
psf_wl_grid = data['psf_wl_grid']

psf_source_offset_x_px = data['psf_source_offset_x_px']
psf_source_offset_y_px = data['psf_source_offset_y_px']
psf_source_offset_x_asec = data['psf_source_offset_x_asec']
psf_source_offset_y_asec = data['psf_source_offset_y_asec']
psf_center_os = (data['center_x_os'], data['center_y_os'])
psf_center_det = (data['center_x_det'], data['center_y_det'])
psf_oversampling = data['oversampling_factor']

del data